# 🛰️ HỆ THỐNG TRUYỀN DỮ LIỆU QUA VỆ TINH IRIDIUM - ROCKBLOCK 9603
## 📥 MODULE NHẬN & PHẢN HỒI (RECEIVER & AUTO-REPLY) - SERIAL / ID: 0235708

### 📖 Cơ chế nhận và tự động phản hồi (Auto-ACK / 2-way):
* **Nguyên lý nhận:**
  1. RockBLOCK **0235707** (Sender) gửi bản tin lên trạm vệ tinh Iridium.
  2. Thiết bị nhận **0235708** gọi lệnh `AT+SBDIX` để kiểm tra hộp thư (Mailbox check).
  3. Nếu `MT Status == 1`, tin nhắn được tải từ Gateway về bộ đệm MT của modem và đọc ra bằng `AT+SBDRT`.
* **Nguyên lý tự động gửi lại 1 bản tin phản hồi (ACK):**
  4. Ngay khi đọc tin thành công, hệ thống **tự động nạp bản tin phản hồi (ACK)** vào bộ đệm MO bằng lệnh `AT+SBDWT`.
  5. Gọi tiếp lệnh `AT+SBDIX` để bắn bản tin phản hồi ngược lại cho thiết bị gửi **0235707**.
  6. Hỗ trợ cả 2 cơ chế định tuyến:
     * **Cách 1 (Direct Addressing):** Tự động ghép tiền tố `RB0235707` vào bản tin phản hồi.
     * **Cách 2 (SBD_ROCKBLOCK):** Gửi bản tin thô (Gateway chuyển tiếp cố định theo IMEI cấu hình trên web).

--- 
### ⚙️ Các lệnh AT cốt lõi sử dụng trong Notebook nhận & phản hồi:
* `AT`: Kiểm tra modem phản hồi (OK).
* `AT+CSQ`: Kiểm tra mức sóng vệ tinh Iridium.
* `AT+SBDIX`: Thăm dò hộp thư vệ tinh (Mailbox check) và kích hoạt truyền tin.
* `AT+SBDRT`: Đọc nội dung tin nhắn dạng văn bản (Text) từ bộ đệm MT.
* `AT+SBDD1`: Xóa bộ đệm MT sau khi đọc.
* `AT+SBDWT`: Nạp nội dung phản hồi (ACK) vào bộ đệm MO để gửi lại.
* `AT+SBDD0`: Xóa bộ đệm MO sau khi gửi.


In [ ]:
# CELL 1: KHAI BÁO THƯ VIỆN VÀ CẤU HÌNH THIẾT BỊ
import time
import serial
import serial.tools.list_ports
from datetime import datetime

# ==============================================================================
# DANH SÁCH CỔNG SERIAL HIỆN TẠI TRÊN MÁY TÍNH
# ==============================================================================
print("🔍 Đang quét các cổng COM khả dụng trên máy:")
ports = list(serial.tools.list_ports.comports())
if not ports:
    print("  [!] Không tìm thấy cổng COM nào. Hãy kiểm tra kết nối cáp USB-FTDI!")
else:
    for p in ports:
        print(f"  -> Cổng: {p.device:<8} | Mô tả: {p.description}")

# ==============================================================================
# CẤU HÌNH THÔNG SỐ RECEIVER (0235708) & TỰ ĐỘNG PHẢN HỒI (AUTO-REPLY)
# ==============================================================================
PORT = "COM16"                  # Cổng COM của module nhận 0235708
BAUDRATE = 19200                # Tốc độ baud chuẩn của RockBLOCK 9603 là 19200
TIMEOUT = 3                     # Thời gian timeout đọc serial (giây)
LOG_FILE = f"receiver_0235708_{datetime.now().strftime('%Y%m%d_%H%M%S')}.txt"

# --- CẤU HÌNH PHẢN HỒI (AUTO-REPLY / ACK) VỀ SENDER ---
TARGET_SENDER_SERIAL = "0235707" # Serial thiết bị gửi ban đầu để phản hồi về
AUTO_REPLY = True               # True: Tự động gửi lại bản tin khi nhận được tin mới
USE_RB_PREFIX = True            # True nếu dùng Cách 1 (RB0235707...) | False nếu dùng Cách 2 (SBD_ROCKBLOCK trên web)
ACK_PREFIX = "ACK_OK"           # Tiền tố của thông điệp phản hồi

print(f"\n⚙️ Cấu hình thiết bị nhận: Port = {PORT}, Baud = {BAUDRATE}")
print(f"🎯 Đích phản hồi về    : {TARGET_SENDER_SERIAL} (Tiền tố: RB{int(TARGET_SENDER_SERIAL):07d} nếu bật Prefix)")
print(f"🤖 Tự động gửi lại tin : {'BẬT (Enabled)' if AUTO_REPLY else 'TẮT (Disabled)'}")
print(f"🏷️ Định dạng phản hồi  : {'Cách 1: RB-to-RB Prefix' if USE_RB_PREFIX else 'Cách 2: SBD_ROCKBLOCK Raw'}")
print(f"📄 File lưu log        : {LOG_FILE}")


In [ ]:
# CELL 2: ĐỊNH NGHĨA CLASS ĐIỀU KHIỂN ROCKBLOCK RECEIVER & AUTO-REPLY
class RockBlockReceiver:
    """Lớp điều khiển module RockBLOCK nhận tín hiệu MT và tự động gửi lại phản hồi MO qua vệ tinh Iridium."""
    
    MT_STATUS_DESC = {
        0: "Không có tin nhắn MT nào chờ trong hộp thư Gateway.",
        1: "Thành công: Đã nhận được 1 tin nhắn MT từ Gateway về bộ đệm modem!",
        2: "Lỗi trong quá trình nhận tin nhắn MT từ Gateway."
    }

    MO_STATUS_DESC = {
        0: "Thành công: MO message đã được trạm vệ tinh Iridium tiếp nhận hoàn tất!",
        1: "Thành công: MO message gửi thành công, nhưng kích thước quá lớn.",
        2: "Thành công: MO message gửi thành công, nhưng không định vị được trạm.",
        32: "Thất bại: Không bắt được sóng vệ tinh Iridium hoặc tín hiệu quá yếu (Timeout).",
        33: "Thất bại: Mất kết nối vô tuyến trong quá trình truyền dữ liệu.",
        34: "Thất bại: Giao thức mạng vệ tinh báo bận / nghẽn kênh.",
        35: "Thất bại: Modem Iridium bị khóa hoặc SIM chưa đăng ký gói cước SBD."
    }

    def __init__(self, port, baudrate=19200, timeout=3, log_file=None):
        self.port = port
        self.baudrate = baudrate
        self.timeout = timeout
        self.log_file = log_file
        self.ser = None
        self.imei = None

    def log(self, message):
        """In log ra console và lưu vào file log với timestamp."""
        now = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        log_entry = f"[{now}] {message}"
        print(log_entry)
        if self.log_file:
            with open(self.log_file, "a", encoding="utf-8") as f:
                f.write(log_entry + "\n")

    def connect(self):
        """Mở cổng Serial kết nối tới RockBLOCK."""
        try:
            self.ser = serial.Serial(
                port=self.port,
                baudrate=self.baudrate,
                bytesize=serial.EIGHTBITS,
                parity=serial.PARITY_NONE,
                stopbits=serial.STOPBITS_ONE,
                timeout=self.timeout
            )
            self.log(f"✅ Đã kết nối thành công tới cổng {self.port} @ {self.baudrate} baud.")
            return True
        except serial.SerialException as e:
            self.log(f"❌ Lỗi mở cổng Serial: {e}")
            return False

    def disconnect(self):
        """Đóng kết nối Serial an toàn."""
        if self.ser and self.ser.is_open:
            self.ser.close()
            self.log("🔌 Đã ngắt kết nối cổng Serial.")

    def send_cmd(self, cmd, wait=1.5):
        """Gửi lệnh AT tới modem và đọc phản hồi."""
        if not self.ser or not self.ser.is_open:
            self.log("❌ Cổng Serial chưa mở!")
            return ""
        
        self.ser.reset_input_buffer()
        self.ser.write((cmd + "\r").encode("ascii"))
        time.sleep(wait)
        
        response = self.ser.read_all().decode("ascii", errors="ignore").strip()
        return response

    def init_modem(self):
        """Khởi tạo cấu hình ban đầu cho RockBLOCK."""
        self.log("--- Bắt đầu khởi tạo cấu hình modem bên nhận ---")
        resp = self.send_cmd("AT", wait=1)
        if "OK" not in resp:
            self.log(f"⚠️ Modem không phản hồi lệnh AT: {resp}")
            return False
            
        self.send_cmd("ATE0", wait=1)
        self.send_cmd("AT&K0", wait=1)
        
        fw = self.send_cmd("AT+CGMR", wait=1.5)
        self.imei = self.send_cmd("AT+CGSN", wait=1.5).replace("OK", "").strip()
        
        self.log(f"📱 IMEI thiết bị nhận: {self.imei}")
        self.log(f"💾 Firmware          : {fw.replace('OK', '').strip()}")
        self.log("✅ Modem nhận đã sẵn sàng!")
        return True

    def check_signal(self):
        """Kiểm tra mức sóng vệ tinh Iridium (AT+CSQ). Trả về 0-5 vạch."""
        resp = self.send_cmd("AT+CSQ", wait=2)
        if "+CSQ:" in resp:
            try:
                bars_str = resp.split("+CSQ:")[1].strip().split()[0]
                bars = int(bars_str)
                bar_display = "■" * bars + "□" * (5 - bars)
                self.log(f"📶 Mức tín hiệu vệ tinh: [{bar_display}] ({bars}/5 vạch)")
                return bars
            except Exception:
                pass
        self.log(f"⚠️ Không đọc được mức sóng. Phản hồi: {resp}")
        return 0

    def read_buffer_content(self):
        """Đọc nội dung tin nhắn văn bản từ MT Buffer bằng lệnh AT+SBDRT, sau đó xóa buffer bằng AT+SBDD1."""
        self.log("📖 Đang đọc nội dung từ MT Buffer (AT+SBDRT)...")
        raw_msg = self.send_cmd("AT+SBDRT", wait=1.5)
        
        cleaned_msg = raw_msg
        lines = [line.strip() for line in raw_msg.splitlines() if line.strip() and line.strip() != "OK"]
        if lines:
            cleaned_msg = "\n".join(lines)

        self.log("=" * 60)
        self.log("📩 NỘI DUNG TIN NHẮN NHẬN ĐƯỢC:")
        self.log(f"👉 \"{cleaned_msg}\"")
        self.log("=" * 60)

        self.send_cmd("AT+SBDD1", wait=1)
        return cleaned_msg

    def check_mailbox(self, wait_sbd=20):
        """
        Kiểm tra hộp thư Iridium bằng lệnh AT+SBDIX.
        Trả về: (has_message: bool, message_text: str hoặc None, queued_count: int)
        """
        self.log("📡 Đang gửi lệnh AT+SBDIX để kiểm tra hộp thư vệ tinh (Mailbox Check)...")
        sbdix_resp = self.send_cmd("AT+SBDIX", wait=wait_sbd)
        self.log(f"📥 Phản hồi phiên SBDIX:\n{sbdix_resp}")

        if "+SBDIX:" in sbdix_resp:
            try:
                params = sbdix_resp.split("+SBDIX:")[1].strip().split(",")
                mt_status = int(params[2].strip())
                mtmsn = int(params[3].strip())
                mt_len = int(params[4].strip())
                mt_queued = int(params[5].strip())

                desc = self.MT_STATUS_DESC.get(mt_status, f"Mã trạng thái MT không xác định: {mt_status}")
                self.log(f"📊 Kết quả MT Status = {mt_status}: {desc}")
                self.log(f"🔢 MTMSN: {mtmsn} | Độ dài: {mt_len} bytes | Còn trong hàng đợi: {mt_queued}")

                if mt_status == 1:
                    msg_text = self.read_buffer_content()
                    return True, msg_text, mt_queued
                else:
                    return False, None, mt_queued

            except Exception as e:
                self.log(f"❌ Lỗi giải mã phản hồi SBDIX: {e}")
                return False, None, 0
        else:
            self.log("⚠️ Không nhận được phản hồi +SBDIX từ modem.")
            return False, None, 0

    def send_reply(self, reply_text, target_serial="0235707", use_rb_prefix=True, wait_sbd=20):
        """
        Gửi lại 1 bản tin phản hồi (Reply / ACK) ngược về phía Sender qua vệ tinh.
        - reply_text: Nội dung tin nhắn muốn gửi lại.
        - target_serial: Serial của thiết bị nhận phản hồi (0235707).
        - use_rb_prefix: True nếu ghép tiền tố RB<Serial> (Cách 1), False nếu dùng SBD_ROCKBLOCK (Cách 2).
        """
        if use_rb_prefix and target_serial:
            clean_serial = f"{int(target_serial):07d}"
            payload = f"RB{clean_serial}{reply_text}"
            prefix_info = f"RB{clean_serial}"
        else:
            payload = reply_text
            clean_serial = target_serial or "Default Gateway"
            prefix_info = "None (SBD_ROCKBLOCK)"

        self.log("=" * 60)
        self.log("📤 BẮT ĐẦU GỬI BẢN TIN PHẢN HỒI (REPLY / ACK) VỀ SENDER:")
        self.log(f"🎯 Đích đến          : {clean_serial}")
        self.log(f"🏷️  Tiền tố          : {prefix_info}")
        self.log(f"📝 Nội dung phản hồi : \"{reply_text}\"")
        self.log(f"📦 Chuỗi nạp modem   : \"{payload}\"")
        self.log(f"📏 Độ dài gửi đi     : {len(payload.encode('ascii'))} bytes")
        self.log("=" * 60)

        # 1. Xóa bộ đệm MO cũ
        self.send_cmd("AT+SBDD0", wait=1)

        # 2. Nạp nội dung phản hồi vào MO Buffer
        write_resp = self.send_cmd(f"AT+SBDWT={payload}", wait=1)
        if "OK" not in write_resp:
            self.log(f"❌ Không thể nạp bản tin phản hồi vào MO Buffer: {write_resp}")
            return False
        self.log("✅ Đã nạp bản tin phản hồi vào MO Buffer thành công.")

        # 3. Kích hoạt phiên vệ tinh SBDIX để truyền dữ liệu
        self.log(f"🚀 Đang gọi AT+SBDIX truyền bản tin phản hồi lên vệ tinh (vui lòng đợi {wait_sbd}s)..." )
        sbdix_resp = self.send_cmd("AT+SBDIX", wait=wait_sbd)
        self.log(f"📥 Phản hồi phiên gửi phản hồi:\n{sbdix_resp}")

        # 4. Phân tích kết quả gửi MO
        if "+SBDIX:" in sbdix_resp:
            try:
                params = sbdix_resp.split("+SBDIX:")[1].strip().split(",")
                mo_status = int(params[0].strip())
                momsn = int(params[1].strip())
                
                # Xóa MO Buffer sau khi gửi
                self.send_cmd("AT+SBDD0", wait=1)

                desc = self.MO_STATUS_DESC.get(mo_status, f"MO Status: {mo_status}")
                if mo_status in [0, 1, 2]:
                    self.log(f"✨ KẾT QUẢ: GỬI BẢN TIN PHẢN HỒI THÀNH CÔNG! (MOMSN: {momsn})")
                    return True
                else:
                    self.log(f"⚠️ GỬI PHẢN HỒI THẤT BẠI: Mã lỗi MO = {mo_status} ({desc})")
                    return False
            except Exception as e:
                self.log(f"❌ Lỗi phân tích phản hồi SBDIX: {e}")
                return False
        else:
            self.log("❌ Modem không phản hồi chuỗi +SBDIX.")
            return False

    def listen_loop(self, poll_interval=30, max_cycles=10, auto_reply=True, target_serial="0235707", use_rb_prefix=True, ack_prefix="ACK_OK"):
        """
        Chế độ lắng nghe liên tục và TỰ ĐỘNG PHẢN HỒI:
        - Định kỳ kiểm tra hộp thư Iridium.
        - KHI NHẬN ĐƯỢC TIN NHẮN MỚI: Tự động gửi lại 1 bản tin ACK ngược lại cho Sender!
        """
        self.log("=" * 60)
        self.log(f"🎧 BẬT CHẾ ĐỘ TRỰC NHẬN TIN NHẮN (Chu kỳ thăm dò: {poll_interval}s)")
        if auto_reply:
            self.log(f"🤖 Chế độ tự động phản hồi: BẬT -> Gửi lại bản tin về: {target_serial}")
        self.log("👉 Nhấn nút [Stop / Interrupt Kernel] trong Jupyter để dừng lắng nghe bất cứ lúc nào.")
        self.log("=" * 60)

        cycle = 0
        try:
            while True:
                cycle += 1
                self.log(f"\n--- Chu kỳ kiểm tra {cycle}" + (f"/{max_cycles}" if max_cycles else "") + " ---")
                
                bars = self.check_signal()
                if bars >= 1:
                    has_msg, msg, queued = self.check_mailbox(wait_sbd=20)
                    if has_msg:
                        self.log(f"🎉 ĐÃ NHẬN ĐƯỢC TIN NHẮN MỚI: \"{msg}\"")
                        
                        # TỰ ĐỘNG GỬI LẠI 1 BẢN TIN PHẢN HỒI
                        if auto_reply:
                            timestamp_str = datetime.now().strftime('%H:%M:%S')
                            short_content = msg[:12] if len(msg) > 12 else msg
                            reply_content = f"{ack_prefix}: RECV '{short_content}' TIME={timestamp_str}"
                            
                            self.log("🔄 Đang tự động gửi lại bản tin phản hồi...")
                            time.sleep(2)
                            self.send_reply(
                                reply_text=reply_content,
                                target_serial=target_serial,
                                use_rb_prefix=use_rb_prefix,
                                wait_sbd=20
                            )

                        # Kéo tiếp nếu còn tin trong hàng đợi Gateway
                        while queued > 0:
                            self.log(f"📦 Vẫn còn {queued} tin nhắn trong hàng đợi Gateway. Đang tải tiếp...")
                            time.sleep(3)
                            has_next, next_msg, queued = self.check_mailbox(wait_sbd=20)
                            if has_next and auto_reply:
                                time.sleep(2)
                                rep = f"{ack_prefix}: RECV '{next_msg[:12]}' TIME={datetime.now().strftime('%H:%M:%S')}"
                                self.send_reply(rep, target_serial=target_serial, use_rb_prefix=use_rb_prefix, wait_sbd=20)
                    else:
                        self.log("📭 Hộp thư trống, chưa có tin nhắn mới.")
                else:
                    self.log("⚠️ Sóng vệ tinh 0 vạch. Tạm bỏ qua chu kỳ này để tiết kiệm thời gian.")

                if max_cycles and cycle >= max_cycles:
                    self.log("⏹️ Đã đạt giới hạn chu kỳ kiểm tra.")
                    break

                self.log(f"💤 Nghỉ {poll_interval} giây trước lần kiểm tra kế tiếp...")
                time.sleep(poll_interval)

        except KeyboardInterrupt:
            self.log("🛑 Người dùng đã bấm dừng chế độ lắng nghe.")


In [ ]:
# CELL 3: KHỞI TẠO ĐỐI TƯỢNG VÀ KẾT NỐI VỚI ROCKBLOCK RECEIVER
receiver = RockBlockReceiver(port=PORT, baudrate=BAUDRATE, log_file=LOG_FILE)

# Mở cổng kết nối
if receiver.connect():
    # Cấu hình modem và đọc IMEI
    ready = receiver.init_modem()
    if not ready:
        print("⚠️ Khởi tạo modem thất bại. Vui lòng kiểm tra cổng COM hoặc rút cắm lại thiết bị.")
else:
    print("❌ Không mở được cổng Serial. Hãy kiểm tra lại biến PORT ở Cell 1!")


In [ ]:
# CELL 4: KIỂM TRA SÓNG VỆ TINH
# Đảm bảo anten của RockBLOCK 0235708 được đặt ngoài trời thoáng
print("📡 Kiểm tra mức sóng của thiết bị nhận:")
receiver.check_signal()


In [ ]:
# CELL 5: KIỂM TRA HỘP THƯ 1 LẦN & TỰ ĐỘNG PHẢN HỒI (ONE-SHOT CHECK & AUTO-ACK)
# Kiểm tra xem thiết bị 0235707 đã gửi tin nhắn tới chưa.
# NẾU CÓ TIN NHẮN: Tự động gửi lại 1 bản tin phản hồi về cho 0235707!
print("📬 Đang thực hiện kiểm tra hộp thư (One-shot check)...")
has_message, message_content, remaining_queued = receiver.check_mailbox(wait_sbd=20)

if has_message:
    print(f"\n🎉 NHẬN THÀNH CÔNG: \"{message_content}\"")
    
    # TỰ ĐỘNG GỬI LẠI 1 BẢN TIN PHẢN HỒI (ACK)
    if AUTO_REPLY:
        reply_msg = f"{ACK_PREFIX}: RECV '{message_content[:15]}' TIME={datetime.now().strftime('%H:%M:%S')}"
        print(f"\n📤 [AUTO-REPLY] Đang tự động gửi lại bản tin phản hồi về Sender ({TARGET_SENDER_SERIAL})...")
        ack_ok = receiver.send_reply(
            reply_text=reply_msg,
            target_serial=TARGET_SENDER_SERIAL,
            use_rb_prefix=USE_RB_PREFIX,
            wait_sbd=20
        )
        if ack_ok:
            print(f"🎉 ĐÃ GỬI BẢN TIN PHẢN HỒI THÀNH CÔNG VỀ CHO {TARGET_SENDER_SERIAL}!")
        else:
            print("⚠️ Gửi bản tin phản hồi thất bại. Vui lòng kiểm tra sóng vệ tinh!")
else:
    print("\n📭 Hiện chưa có tin nhắn nào trong hộp thư.")


In [ ]:
# CELL 6: BẬT CHẾ ĐỘ LẮNG NGHE ĐỊNH KỲ VÀ TỰ ĐỘNG PHẢN HỒI (CONTINUOUS LISTENING + AUTO-ACK)
# Chạy vòng lặp tự động thăm dò hộp thư mỗi 30 giây.
# Khi nhận được bất kỳ tin nhắn nào -> TỰ ĐỘNG BẮN LẠI 1 BẢN TIN PHẢN HỒI!
# Bạn có thể bấm nút Stop (Interrupt kernel) trên thanh công cụ Jupyter khi muốn dừng.
receiver.listen_loop(
    poll_interval=30,
    max_cycles=5,
    auto_reply=AUTO_REPLY,
    target_serial=TARGET_SENDER_SERIAL,
    use_rb_prefix=USE_RB_PREFIX,
    ack_prefix=ACK_PREFIX
)


In [ ]:
# CELL 7: ĐÓNG KẾT NỐI KHI HOÀN TẤT
receiver.disconnect()
print(f"📄 Toàn bộ log nhận tin đã được ghi vào file: {LOG_FILE}")
